# Aligned facial keypoint training

Run every cell in order. This notebook reads the original images and CSV directly, rather than the old `all_*.pt` caches. Run it from the project directory containing `data/training` and `data/training_frames_keypoints.csv`.

In [ ]:
import os
import torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from models.naimishnet import NaimishNet
from src.data_pipeline import FacialKeypointsDataset
from src.alignment_workflow import plot_prediction, transfer_backbone

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
dataset = FacialKeypointsDataset('data/training_frames_keypoints.csv', 'data/training')
train_loader = DataLoader(dataset, batch_size=32, shuffle=True)
sample = next(iter(DataLoader(dataset, batch_size=1, shuffle=False)))
plot_batch = (sample['image'], sample['keypoints'], sample['mask'])
os.makedirs('output', exist_ok=True)

## Train the baseline
Both models use masked MSE against the same freshly computed normalized targets. Saved filenames are distinct from the old checkpoints.

In [ ]:
def train(model, epochs, learning_rate):
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        for batch in train_loader:
            images = batch['image'].to(device)
            targets = batch['keypoints'].to(device)
            masks = batch['mask'].to(device)
            optimizer.zero_grad()
            outputs = model(images).reshape(-1, 90, 2)
            loss = ((outputs - targets).square() * masks.unsqueeze(-1)).sum() / (masks.sum().clamp_min(1) * 2)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f'Epoch {epoch + 1}/{epochs}, loss: {total_loss / len(train_loader):.4f}')

baseline = NaimishNet().to(device)
train(baseline, epochs=10, learning_rate=0.001)
torch.save(baseline.state_dict(), 'output/naimishnet_aligned.pth')
plot_prediction(baseline, plot_batch, device, 'Baseline prediction')
plt.show()

## Train BAM + CoordConv
Transfer convolutional weights only, adapting the first layer for the two coordinate channels. The regression head starts fresh. Red points are predictions and green points are matching ground truth; masked landmarks are not plotted.

In [ ]:
advanced = NaimishNet(use_bam_and_coord=True).to(device)
transfer_backbone(baseline, advanced)
train(advanced, epochs=50, learning_rate=0.0005)
torch.save(advanced.state_dict(), 'output/naimishnet_bam_coord_aligned.pth')
plot_prediction(advanced, plot_batch, device, 'BAM + CoordConv prediction')
plt.show()